# Part III — Conditioning and Stability

## Trefethen & Bau, *Numerical Linear Algebra* (1997) — Lecture 12–19

这是《Numerical Linear Algebra》读书笔记的第 3 册。目标不是摘要原书，而是把每一讲整理成一份**可以独立读懂的数值线性代数讲义**，再把它映射到现代 ML systems。

每一讲尽量保持同一结构：数学对象 → 关键公式 → 几何/算法解释 → numerical stability → ML systems mapping → Python experiment。

本册自洽：下面的 setup cell 提供全部依赖，按顺序 run all 即可。

原书 PDF：https://www.stat.uchicago.edu/~lekheng/courses/309/books/Trefethen-Bau.pdf

---

**本系列共 6 册**（Trefethen & Bau, *Numerical Linear Algebra*, 40 Lectures）

| | |
|---|---|
| Part I | [Fundamentals](01_fundamentals.ipynb) |
| Part II | [QR Factorization and Least Squares](02_qr_least_squares.ipynb) |
| Part III | [Conditioning and Stability](03_conditioning_stability.ipynb) |
| Part IV | [Systems of Equations](04_systems_of_equations.ipynb) |
| Part V | [Eigenvalues](05_eigenvalues.ipynb) |
| Part VI | [Iterative Methods](06_iterative_methods.ipynb) |

索引与阅读顺序见 [00_index.ipynb](00_index.ipynb)。


In [1]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import scipy.linalg as sla
    import scipy.sparse.linalg as spla
    SCIPY=True
except Exception:
    SCIPY=False
rng=np.random.default_rng(7)
np.set_printoptions(precision=5,suppress=True)

def relerr(a,b):
    return np.linalg.norm(a-b)/max(np.linalg.norm(b),1e-30)

def stable_rank(A):
    s=np.linalg.svd(A,compute_uv=False)
    return np.sum(s*s)/(s[0]*s[0])

def spectral_norm_power(A,steps=30,seed=0):
    r=np.random.default_rng(seed)
    v=r.normal(size=A.shape[1])
    v/=np.linalg.norm(v)
    for _ in range(steps):
        v=A.T@(A@v)
        v/=np.linalg.norm(v)
    return np.linalg.norm(A@v)

def make_cond(n,kappa,seed=0):
    r=np.random.default_rng(seed)
    Q1,_=np.linalg.qr(r.normal(size=(n,n)))
    Q2,_=np.linalg.qr(r.normal(size=(n,n)))
    s=np.geomspace(1,1/kappa,n)
    return Q1@np.diag(s)@Q2.T

print(f'NumPy {np.__version__} | SciPy {SCIPY}')

NumPy 2.4.2 | SciPy True


## Notation / 贯穿全书的符号

我们主要讨论实矩阵；复数情形把转置 $A^T$ 换成共轭转置 $A^*$。

- $A\in\mathbb{R}^{m\times n}$
- 向量 2-norm：

$$
\Vert x\Vert_2=\sqrt{x^Tx}
$$

- induced matrix 2-norm：

$$
\Vert A\Vert_2=\max_{x\neq0}\frac{\Vert Ax\Vert_2}{\Vert x\Vert_2}=\sigma_{\max}(A)
$$

- Frobenius norm：

$$
\Vert A\Vert_F^2=\sum_{ij}a_{ij}^2=\sum_i\sigma_i^2
$$

- condition number：

$$
\kappa_2(A)=\Vert A\Vert_2\Vert A^{-1}\Vert_2
=\frac{\sigma_{\max}}{\sigma_{\min}}
$$

- unit roundoff：记为 $u$。典型 floating-point model：

$$
\mathrm{fl}(a\circ b)=(a\circ b)(1+\delta),\qquad |\delta|\lesssim u.
$$

计算机算一次加减乘除，不会得到精确的 $a\circ b$，而是得到一个带相对误差的结果。逐项：

- $\mathrm{fl}(\cdot)$：floating-point，机器实际算出来的数；
- $a\circ b$：一次精确运算（$\circ$ 是 $+,-,\times,/$）；
- $\delta$：这次运算引入的相对误差；
- $u$：unit roundoff，这种格式「一次正确舍入」的相对误差上限。

左边是机器结果，右边是「真值再乘 $1+\delta$」。约束的是**相对误差**，不是绝对误差：真值若是 $1.0$，FP32 下次运算大约落在 $1\pm 6\times 10^{-8}$，不会无缘无故错到 $1.01$。

常见 $u$：

- FP32：$u\approx 2^{-24}\approx 6\times 10^{-8}$
- FP16：$u\approx 2^{-11}\approx 5\times 10^{-4}$
- BF16：$u\approx 2^{-8}\approx 4\times 10^{-3}$

一次运算只错 $u$。病态问题可能把这个 $u$ 放大成 $\kappa u$ 量级的解误差。所以不要把「格式很粗」「矩阵很病态」「算法多放大了 rounding」三件事混成一句“数值不稳”。

### 区分

不要把下面三个问题混在一起：

1. **operator amplification**：$\Vert A\Vert$ 大不大？
2. **problem conditioning**：$A^{-1}$ 是否敏感？
3. **algorithm stability**：实现是否额外放大 rounding error？

现代 ML numerics 中，大量争论其实是把这三个层次混在了一起。

# Lecture 12 — Conditioning and Condition Numbers

### 1. Conditioning 是函数的局部 sensitivity

把问题写成

$$
y=f(x).
$$

这里的 $D$ 表示 **derivative（导数）算子**，所以 $Df(x)$ 表示函数 $f$ 在输入 $x$ 处的导数：

- 若 $f:\mathbb{R}\to\mathbb{R}$，则 $Df(x)=f'(x)$，就是普通导数；
- 若 $f:\mathbb{R}^n\to\mathbb{R}^m$，则 $Df(x)$ 是 $m\times n$ 的 Jacobian matrix：

$$
Df(x)=J_f(x)
=\left[\frac{\partial f_i}{\partial x_j}\right].
$$

它的意义来自一阶近似：给输入一个很小的扰动 $\Delta x$，输出变化为

$$
f(x+\Delta x)-f(x)
\approx Df(x)\,\Delta x.
$$

因此局部绝对 condition number 是 Jacobian 的 operator norm：

$$
\kappa_{\mathrm{abs}}(x)
=\Vert Df(x)\Vert
=\max_{\Delta x\ne0}
\frac{\Vert Df(x)\Delta x\Vert}{\Vert\Delta x\Vert}.
$$

它表示：在 $x$ 附近，单位大小的输入扰动最多会造成多大的输出扰动。

例如：

- $f(x)=x^2$：$Df(x)=2x$，所以 $\kappa_{\mathrm{abs}}(x)=|2x|$；
- $f(x)=Ax$：$Df(x)=A$，所以 $\kappa_{\mathrm{abs}}(x)=\Vert A\Vert$。

相对 condition number 再除去输入输出本身的尺度；在 normwise 情形下通常写作

$$
\kappa_{\mathrm{rel}}(x)
=\frac{\Vert Df(x)\Vert\,\Vert x\Vert}{\Vert f(x)\Vert},
$$

前提是 $x\ne0$ 且 $f(x)\ne0$。

### 2. Linear solve 的一阶 perturbation（按原书）

设 $A$ 可逆，原问题为

$$
Ax=b.
$$

原书把 $b$ 的扰动和 $A$ 的扰动分开讨论。

#### 2.1 固定 $A$，只扰动 $b$

右端项由 $b$ 变为 $b+\Delta b$，解由 $x$ 变为 $x+\Delta x$：

$$
A(x+\Delta x)=b+\Delta b.
$$

减去原方程 $Ax=b$：

$$
A\Delta x=\Delta b,
$$

所以

$$
\Delta x=A^{-1}\Delta b.
$$

取 norm：

$$
\Vert\Delta x\Vert
\le
\Vert A^{-1}\Vert\Vert\Delta b\Vert.
$$

除以 $\Vert x\Vert$：

$$
\frac{\Vert\Delta x\Vert}{\Vert x\Vert}
\le
\Vert A^{-1}\Vert
\frac{\Vert\Delta b\Vert}{\Vert x\Vert}.
$$

又因为 $b=Ax$，

$$
\Vert b\Vert\le\Vert A\Vert\Vert x\Vert,
$$

所以

$$
\frac{1}{\Vert x\Vert}
\le
\frac{\Vert A\Vert}{\Vert b\Vert}.
$$

代回便得到

$$
\boxed{
\frac{\Vert\Delta x\Vert}{\Vert x\Vert}
\le
\underbrace{\Vert A\Vert\Vert A^{-1}\Vert}_{\kappa(A)}
\frac{\Vert\Delta b\Vert}{\Vert b\Vert}
}
$$

即

$$
\frac{\Vert\Delta x\Vert}{\Vert x\Vert}
\le
\kappa(A)
\frac{\Vert\Delta b\Vert}{\Vert b\Vert}.
$$

#### 2.2 固定 $b$，只扰动 $A$

矩阵由 $A$ 变为 $A+\Delta A$，解由 $x$ 变为 $x+\Delta x$：

$$
(A+\Delta A)(x+\Delta x)=b.
$$

展开：

$$
Ax+A\Delta x+\Delta A\,x+\Delta A\,\Delta x=b.
$$

利用 $Ax=b$ 消去两边相同的项：

$$
A\Delta x+\Delta A\,x+\Delta A\,\Delta x=0.
$$

这里采用原书的 **infinitesimal / first-order perturbation** 分析：$\Delta A$ 和 $\Delta x$ 都是一阶小量，所以它们的乘积 $\Delta A\,\Delta x$ 是二阶小量，忽略它：

$$
A\Delta x+\Delta A\,x\approx0.
$$

因此

$$
\Delta x\approx-A^{-1}\Delta A\,x.
$$

取 norm 并除以 $\Vert x\Vert$：

$$
\frac{\Vert\Delta x\Vert}{\Vert x\Vert}
\lesssim
\Vert A^{-1}\Vert\Vert\Delta A\Vert.
$$

乘除一个 $\Vert A\Vert$：

$$
\boxed{
\frac{\Vert\Delta x\Vert}{\Vert x\Vert}
\lesssim
\underbrace{\Vert A\Vert\Vert A^{-1}\Vert}_{\kappa(A)}
\frac{\Vert\Delta A\Vert}{\Vert A\Vert}
}
$$

即

$$
\frac{\Vert\Delta x\Vert}{\Vert x\Vert}
\lesssim
\kappa(A)
\frac{\Vert\Delta A\Vert}{\Vert A\Vert}.
$$

#### 2.3 为什么两个一阶影响可以相加

> **Note（全微分观点）**：将线性系统的解视为关于矩阵 $A$ 和右端项 $b$ 的二元函数
> $$
> x=F(A,b)=A^{-1}b.
> $$
> 当 $A$ 与 $b$ 同时发生小扰动时，解的一阶变化由 $F$ 的全微分给出。

当 $A$ 和 $b$ 同时发生小扰动时，

$$
(A+\Delta A)(x+\Delta x)=b+\Delta b.
$$

展开并利用 $Ax=b$：

$$
A\Delta x+\Delta A\,x+\Delta A\,\Delta x=\Delta b.
$$

在一阶分析中，$\Delta A\,\Delta x$ 是两个小量的乘积，属于二阶项，因此忽略：

$$
A\Delta x+\Delta A\,x\approx\Delta b.
$$

解出 $\Delta x$：

$$
\boxed{
\Delta x
\approx
\underbrace{-A^{-1}\Delta A\,x}_{A\text{ 的扰动造成的变化}}
+
\underbrace{A^{-1}\Delta b}_{b\text{ 的扰动造成的变化}}
}
$$

这就是解映射 $F(A,b)$ 的全微分：

$$
DF(A,b)[\Delta A,\Delta b]
=
D_AF(A,b)[\Delta A]
+
D_bF(A,b)[\Delta b].
$$

**两个影响能够相加，是因为导数（线性化）对扰动是线性的。** 这不是说两个误差的大小一定直接相加，而是说一阶的**误差向量**相加。

接着取 norm。由三角不等式，

$$
\begin{aligned}
\Vert\Delta x\Vert
&\lesssim
\Vert A^{-1}\Delta A\,x\Vert
+
\Vert A^{-1}\Delta b\Vert \\
&\le
\Vert A^{-1}\Vert\Vert\Delta A\Vert\Vert x\Vert
+
\Vert A^{-1}\Vert\Vert\Delta b\Vert.
\end{aligned}
$$

除以 $\Vert x\Vert$：

$$
\frac{\Vert\Delta x\Vert}{\Vert x\Vert}
\lesssim
\Vert A^{-1}\Vert\Vert\Delta A\Vert
+
\Vert A^{-1}\Vert
\frac{\Vert\Delta b\Vert}{\Vert x\Vert}.
$$

利用前两节已经推过的两个换算，

$$
\Vert A^{-1}\Vert\Vert\Delta A\Vert
=
\kappa(A)\frac{\Vert\Delta A\Vert}{\Vert A\Vert},
$$

以及由 $\Vert b\Vert\le\Vert A\Vert\Vert x\Vert$ 得到

$$
\Vert A^{-1}\Vert
\frac{\Vert\Delta b\Vert}{\Vert x\Vert}
\le
\kappa(A)
\frac{\Vert\Delta b\Vert}{\Vert b\Vert},
$$

最终得到

$$
\boxed{
\frac{\Vert\Delta x\Vert}{\Vert x\Vert}
\lesssim
\kappa(A)
\left(
\frac{\Vert\Delta A\Vert}{\Vert A\Vert}
+
\frac{\Vert\Delta b\Vert}{\Vert b\Vert}
\right).
}
$$

这里括号里的加号来自**三角不等式**，所以它是最坏情况 upper bound。两个误差向量也可能方向相反、部分抵消；此时实际误差会比这个 bound 小。

#### 2.4 这条合并公式的使用条件

1. $A$ 必须可逆；
2. $A$ 和 $b$ 的扰动必须足够小，使一阶线性化有效；
3. 特别地，应有
   $$
   \Vert A^{-1}\Delta A\Vert\ll1,
   $$
   常用的保守判据是
   $$
   \kappa(A)\frac{\Vert\Delta A\Vert}{\Vert A\Vert}\ll1;
   $$
4. 使用的是与向量 norm 相容的 induced matrix norm；
5. 写相对误差还要求 $b\ne0$、$x\ne0$。

如果这些条件不成立，尤其是 $\kappa(A)\Vert\Delta A\Vert/\Vert A\Vert$ 已接近 $1$，一阶公式就不能再被信任，需要有限扰动分析。当前 Lecture 12 只掌握上述全微分观点即可。

这就是 condition number 的工程意义：输入相对误差的最坏放大倍数由 $\kappa(A)$ 控制；实际放大取决于扰动方向。

### 3. SVD interpretation

$$
\Vert A^{-1}\Vert_2=1/\sigma_{\min}(A),
$$

所以最危险的是 near-null direction。

### 4. ML mapping

欠约束的 representation、近冗余 feature、flat Hessian directions，本质上都在制造小 singular/eigen directions。

In [2]:
A=make_cond(30,1e8,10)
xt=rng.normal(size=30)
b=A@xt
x0=np.linalg.solve(A,b)
db=rng.normal(size=30)
db*=1e-8*np.linalg.norm(b)/np.linalg.norm(db)
x1=np.linalg.solve(A,b+db)
rb=np.linalg.norm(db)/np.linalg.norm(b)
rx=relerr(x1,x0)
print(f'cond {np.linalg.cond(A)} relative b perturbation {rb} relative x change {rx} amplification {rx / rb}')

cond 100000000.1162295 relative b perturbation 1e-08 relative x change 0.012095175083658887 amplification 1209517.5083658886


**ML numerics 自测**

**Q1.** condition number 本质在量什么？

**A.** 问题 $y=f(x)$ 的局部灵敏度。绝对条件数是 $\Vert Df(x)\Vert$；相对条件数再把输入输出 scale 算进去。

**Q2.** 怎样从 perturbation bound 看出 $\kappa(A)$ 是误差放大倍数？

**A.** 先明确这里所谓的“输入误差”和“输出误差”。定义

$$
\varepsilon_{\mathrm{in}}
:=
\underbrace{\frac{\Vert\Delta A\Vert}{\Vert A\Vert}}_{A\text{ 的相对误差}}
+
\underbrace{\frac{\Vert\Delta b\Vert}{\Vert b\Vert}}_{b\text{ 的相对误差}},
$$

以及

$$
\varepsilon_{\mathrm{out}}
:=
\frac{\Vert\Delta x\Vert}{\Vert x\Vert}.
$$

一阶 bound 就可以重写成

$$
\varepsilon_{\mathrm{out}}
\lesssim
\kappa(A)\varepsilon_{\mathrm{in}},
$$

即

$$
\frac{\varepsilon_{\mathrm{out}}}
{\varepsilon_{\mathrm{in}}}
\lesssim
\kappa(A).
$$

左边才是“相对输出误差 ÷ 相对输入误差”，也就是误差放大倍数。因此准确说法是：**若把两个输入扰动的相对大小之和定义为 $\varepsilon_{\mathrm{in}}$，那么一阶相对解误差的最坏放大倍数不超过 $\kappa(A)$。**

例如两个输入误差分别为 $10^{-6}$ 和 $2\times10^{-6}$，则 $\varepsilon_{\mathrm{in}}=3\times10^{-6}$；若 $\kappa(A)=100$，bound 给出 $\varepsilon_{\mathrm{out}}\lesssim3\times10^{-4}$。

这不表示实际误差一定等于该值：两项可能没有对准最敏感方向，甚至可能互相抵消。它只是 first-order worst-case upper bound。若改用 $\max(\Vert\Delta A\Vert/\Vert A\Vert,\Vert\Delta b\Vert/\Vert b\Vert)$ 定义联合输入误差，则相应 bound 会多一个至多为 $2$ 的常数因子；所以谈“放大倍数”之前必须先说明采用哪一种输入误差度量。

**Q3.** SVD 下最危险的方向是哪条？

**A.** $\Vert A^{-1}\Vert_2=1/\sigma_{\min}$，near-null direction。欠约束 feature / flat Hessian 都在制造这种方向。

**Q4.** 监控 $\kappa$ 是在监控算法，还是监控问题？

**A.** 监控问题本身。算法再 stable，也只能保证 backward error 是 $O(u)$；forward error 仍可到 $\kappa u$。


# Lecture 13 — Floating Point Arithmetic

### 1. Floating-point model

一个 precision 为 $p$ 的 normalized binary floating-point number 可以写成

$$
x=\pm(1.b_1b_2\ldots b_{p-1})_2\,2^e.
$$

这里 $p$ 包含最前面隐含的那个 $1$。有限 significand 意味着相邻 representable numbers 之间存在间距，而且间距会随 exponent 改变。

#### FP32 的 unit roundoff 从哪里来

IEEE FP32（binary32）包含：

- 1 个 sign bit；
- 8 个 exponent bits；
- 23 个显式 fraction bits。

对 normalized number，最前面的 $1$ 不需要存储，因此实际 precision 是

$$
p=1+23=24\ \text{bits}.
$$

在 $1$ 附近，更准确地说是在 binade $[1,2)$ 中，FP32 数写成

$$
(1.b_1b_2\ldots b_{23})_2.
$$

所以 $1$ 之后的下一个可表示数是

$$
1+2^{-23}.
$$

二者间距为

$$
\operatorname{ulp}(1)=2^{-23}
\approx1.1920929\times10^{-7}.
$$

如果采用 round-to-nearest，任意实数最多被舍入半个间距，因此 $1$ 附近的最大绝对舍入误差是

$$
\frac12\operatorname{ulp}(1)
=2^{-24}.
$$

由于此时数值尺度约为 $1$，绝对误差和相对误差数值相同，于是 FP32 的 unit roundoff 为

$$
\boxed{
u=2^{-24}\approx5.9604645\times10^{-8}.
}
$$

所以答案是：**unit roundoff 的确可以从 $1$ 附近推导出来。** 但它不是只在 $1$ 附近才有效。在一般 binade $[2^e,2^{e+1})$ 中，间距变成 $2^{e-23}$，半个间距是 $2^{e-24}$；再除以数值尺度 $2^e$，相对误差仍约为 $2^{-24}$。

#### `eps` 和 unit roundoff 不要混淆

NumPy 给出的

```python
np.finfo(np.float32).eps
```

是 $1$ 与下一个更大 FP32 数的距离：

$$
\mathrm{eps}=2^{-23}\approx1.1920929\times10^{-7}.
$$

数值分析中采用 round-to-nearest 时的 unit roundoff 是其一半：

$$
u=\frac{\mathrm{eps}}{2}=2^{-24}\approx5.96\times10^{-8}.
$$

有些资料把 `machine epsilon` 一词用于 $2^{-23}$，另一些资料把它用于 $2^{-24}$，所以看到该术语时必须检查作者的定义。本讲统一使用：

- `eps`：$1$ 到下一个可表示数的间距；
- $u$：正确舍入的最大相对误差，即 half-ulp bound。

通常用 unit roundoff $u$ 表示一次正确 rounding 的相对误差规模。对一个已经算出来的实数做舍入：

$$
\mathrm{fl}(x)=x(1+\delta),\qquad |\delta|\le u.
$$

对一次精确算术运算，标准模型写成

$$
\mathrm{fl}(a\circ b)=(a\circ b)(1+\delta),\qquad |\delta|\lesssim u.
$$

$\mathrm{fl}(\cdot)$ 是机器实际写出的数，$a\circ b$ 是精确加减乘除，$\delta$ 是这次运算的相对误差。约束的是相对误差，不是绝对误差。

常见 $u$：FP32 $\approx 6\times10^{-8}$，FP16 $\approx 5\times10^{-4}$，BF16 $\approx 4\times10^{-3}$。一次运算只引入 $u$；若问题的 $\kappa$ 很大，解的 forward error 可以到 $\kappa u$ 量级。算法 stable 只保证「没有把 rounding 额外放大」，不保证病态问题仍有精确解。

> **适用范围：** 上面的相对误差模型针对正常范围内的 normalized numbers，并假设没有 overflow/underflow。进入 subnormal 区域后，固定的相对误差界不再成立。

### 2. Catastrophic cancellation

若 $x\approx y$，计算

$$
z=x-y
$$

时，输入本身的微小相对误差可能在小 residual 中占很大比例。

例如本来有

$$
x=1.0000001,\quad y=1.0000000,
$$

差值只有 $10^{-7}$。如果低精度根本无法分辨这两个数，结果直接变成 0。

### 3. BF16 vs FP16 的不同风险

- FP16：mantissa 相对多，但 exponent range 小，容易 overflow/underflow；
- BF16：exponent 类似 FP32，但 mantissa 很粗，容易丢掉小 relative differences。

### 4. ML high-risk operations

- long reduction；
- variance / RMSNorm statistics；
- softmax logits；
- reciprocal / sqrt；
- Gram matrices；
- small-pivot factorization。

In [3]:
for dt in [np.float16,np.float32,np.float64]:
    f=np.finfo(dt)
    print(f'{dt.__name__} eps {f.eps} tiny {f.tiny} max {f.max}')

float16 eps 0.0009765625 tiny 6.103515625e-05 max 65504.0
float32 eps 1.1920928955078125e-07 tiny 1.1754943508222875e-38 max 3.4028234663852886e+38
float64 eps 2.220446049250313e-16 tiny 2.2250738585072014e-308 max 1.7976931348623157e+308


**ML numerics 自测**

**Q1.** $\mathrm{fl}(a\circ b)=(a\circ b)(1+\delta)$ 约束的是什么？

**A.** 一次运算的相对误差 $|\delta|\lesssim u$，不是绝对误差。$u$：FP32 $\approx 6\times 10^{-8}$，FP16 $\approx 5\times 10^{-4}$，BF16 $\approx 4\times 10^{-3}$。

**Q2.** 什么时候相对误差模型会突然崩？

**A.** catastrophic cancellation：$x\approx y$ 时 $x-y$ 把已有误差抬成主导项。低精度甚至根本分不出 $1.0000001$ 和 $1$。

**Q3.** BF16 和 FP16 的风险有何不同？

**A.** FP16 mantissa 较多但 exponent 窄，怕 overflow/underflow；BF16 exponent 像 FP32，但 mantissa 粗，怕丢掉小相对差。

**Q4.** ML 里哪些 op 最吃 $u$？

**A.** 长 reduction、RMSNorm/variance、softmax、reciprocal/sqrt、Gram、$A^TA$、小 pivot 分解。


# Lecture 14 — Stability

### 1. Forward error

算法输出 $\hat y$，真正答案 $y=f(x)$：

$$
\text{forward error}=\Vert \hat y-y\Vert.
$$

### 2. Backward error

找最小 $\Delta x$，使

$$
\hat y=f(x+\Delta x).
$$

如果所需 $\Delta x$ 与 machine precision 同量级，我们说算法 backward stable。

### 3. Residual 不是 forward error

对 linear solve，

$$
r=b-A\hat x.
$$

因为

$$
A(x-\hat x)=r,
$$

所以

$$
x-\hat x=A^{-1}r.
$$

因此

$$
\Vert x-\hat x\Vert \le \Vert A^{-1}\Vert \Vert r\Vert.
$$

如果 $A^{-1}$ 很大，小 residual 仍然可能对应大 solution error。

#### ML training 中的对应关系

上面的推导采用方阵线性系统 $Ax=b$ 的视角。在线性回归或 linear probe 中，通常写成

$$
y=Xw+b_{\mathrm{bias}},
$$

其中 $X\in\mathbb{R}^{m\times n}$ 是 design / feature matrix，$w\in\mathbb{R}^n$ 是待求参数。移去 bias 后，

$$
y-b_{\mathrm{bias}}=Xw.
$$

因此这里的对应关系是

$$
A\longleftrightarrow X,
\qquad
x\longleftrightarrow w,
\qquad
b\longleftrightarrow y-b_{\mathrm{bias}}.
$$

注意乘法顺序。因为代码形式是 `y = X @ w + b_bias`，所以若 $X$ 恰好是方阵且可逆，应写

$$
w=X^{-1}(y-b_{\mathrm{bias}}),
$$

而不是 $(y-b_{\mathrm{bias}})X^{-1}$。

实际训练中通常 $m>n$，$X$ 是长方形矩阵，不存在普通逆矩阵。这时求的是 least-squares 解

$$
\hat w
=\arg\min_w\Vert Xw-(y-b_{\mathrm{bias}})\Vert_2,
$$

理论上写作

$$
\hat w=X^\dagger(y-b_{\mathrm{bias}}),
$$

实现上应使用 QR 或 SVD，而不是显式形成 $X^\dagger$ 或 $(X^TX)^{-1}$。

此时控制参数敏感度的是

$$
\Vert X^\dagger\Vert_2
=\frac{1}{\sigma_{\min}(X)}.
$$

这里还有一个与方阵情形的重要区别。若训练数据含噪声，最优 least-squares residual

$$
r_*=y-b_{\mathrm{bias}}-Xw_*
$$

一般不为零，因此不能直接写成 $w_*-\hat w=X^\dagger r_*$。若比较精确的 least-squares 解 $w_*$ 与数值算法算出的 $\hat w$，应比较两者的 residual：

$$
r_{\hat w}-r_*
=X(w_*-\hat w),
$$

从而

$$
\Vert w_*-\hat w\Vert_2
\le
\Vert X^\dagger\Vert_2
\Vert r_{\hat w}-r_*\Vert_2.
$$

若 $X$ 有很小的 singular value，则某些 $\Delta w$ 可以很大，但 $X\Delta w$ 仍然很小。因此两个 checkpoint 可能具有几乎相同的 prediction 和 loss，却得到差异很大的参数 $w$。这正是 near-null directions 上的 parameter non-identifiability。

若 bias 也需要一同求解，可把常数列并入 design matrix：

$$
\widetilde X=[X\ \mathbf 1],
\qquad
\theta=\begin{bmatrix}w\\b_{\mathrm{bias}}\end{bmatrix},
\qquad
\widetilde X\theta=y.
$$

### 4. Deployment parity mapping

只看 tensor residual/output diff 不够。至少要知道：

- local error 多大；
- downstream operator gain 多大；
- task output 对该方向 sensitivity 多大。

In [4]:
A=make_cond(25,1e12,12)
xt=rng.normal(size=25)
b=A@xt
x=np.linalg.solve(A,b)
print(f'relative residual {relerr(A @ x, b)}')
print(f'forward error {relerr(x, xt)}')
print(f'cond {np.linalg.cond(A)}')

relative residual 1.7607364806733918e-16
forward error 9.214615717784304e-06
cond 1000001499714.1288


**ML numerics 自测**

**Q1.** forward error 和 backward error 各是什么？

**A.** forward：$\Vert\hat y-y\Vert$，答案错多少。backward：最小 $\Delta x$ 使 $\hat y=f(x+\Delta x)$。$\Delta x$ 与 $u$ 同量级就称 backward stable。

**Q2.** 为什么 residual 不是 forward error？

**A.** $r=b-A\hat x$ 时 $x-\hat x=A^{-1}r$，故 $\Vert x-\hat x\Vert\le\Vert A^{-1}\Vert\Vert r\Vert$。$A^{-1}$ 大时，小 residual 仍可对应大解误差。

**Q3.** 只看 output tensor diff 够不够做 deployment parity？

**A.** 不够。还要看 local error、downstream gain、以及 task 对该方向的 sensitivity。

**Q4.** stable 算法保证的是哪一种 error？

**A.** 保证的是小 backward error，不是小 forward error。


# Lecture 15 — More on Stability

### 1. “Backward stable + well-conditioned = accurate”

这是 numerical analysis 最重要的组合关系之一。

若算法 backward stable，意味着它实际上求解的是 nearby problem：

$$
\hat y=f(x+\Delta x),
\qquad
\frac{\Vert \Delta x\Vert}{\Vert x\Vert}=O(u).
$$

若问题的 relative condition number 为 $\kappa$，则一阶近似：

$$
\frac{\Vert \hat y-y\Vert}{\Vert y\Vert}
=O(\kappa u).
$$

所以误差可以拆成：

$$
\boxed{\text{problem sensitivity}}
\times
\boxed{\text{algorithm perturbation}}.
$$

### 2. 为什么这个分解适合 mixed precision

假设某个 kernel 引入局部 perturbation $\epsilon$，下游 Jacobian gain 为 $G$：

$$
\delta y\approx G\epsilon.
$$

这就是现代版的 condition × backward error 思维。

### 3. 一个有用的 sanity scale

若 FP32 $u\sim10^{-7}$，而 $\kappa\sim10^8$，那么 $\kappa u$ 已经接近 10。此时“算法很 stable”也不能保证 forward answer 有意义。

**ML numerics 自测**

**Q1.** “backward stable + well-conditioned = accurate” 怎么拆？

**A.** 算法给出 nearby problem：$\Vert\Delta x\Vert/\Vert x\Vert=O(u)$。问题相对条件数为 $\kappa$，则 $\Vert\hat y-y\Vert/\Vert y\Vert=O(\kappa u)$。

**Q2.** mixed precision 里这句话变成什么？

**A.** kernel 引入局部扰动 $\epsilon$，下游 Jacobian gain 为 $G$，则 $\delta y\approx G\epsilon$。仍是 condition $\times$ backward error。

**Q3.** FP32、$u\sim 10^{-7}$、$\kappa\sim 10^8$ 说明什么？

**A.** $\kappa u$ 已接近 10。“算法很 stable”也不能保证 forward answer 有意义。

**Q4.** 该分别报告哪两个盒子？

**A.** problem sensitivity 和 algorithm perturbation。混成一句“数值不稳”会选错修法。


# Lecture 16 — Stability of Householder Triangularization

### 1. Householder QR 的 backward stability 结论

设 floating-point 算法计算出 $\hat Q,\hat R$。核心结论可理解为存在一个很小的 $\Delta A$，使

$$
A+\Delta A=\hat Q\hat R,
\qquad
\frac{\Vert \Delta A\Vert}{\Vert A\Vert}=O(u)
$$

（更精确的 bound 还含维度常数）。

也就是说算法没有把 rounding error 变成一个远离原输入的问题。

### 2. 为什么 orthogonal transformations 有优势

每一步 reflector $H_k$ 的 norm 是 1：

$$
\Vert H_k\Vert_2=1.
$$

所以前面某一步产生的 perturbation 在后续正交变换中不会被指数放大。

### 3. 但 QR stable ≠ least squares 一定 accurate

如果 $A$ 接近 rank deficient，$R$ 的 diagonal 会出现极小值。QR 可能准确地揭示这个坏条件数，但 solve 本身仍然 sensitive。

**ML numerics 自测**

**Q1.** Householder QR 的 backward stability 结论是什么？

**A.** 存在 $\Delta A$ 使 $A+\Delta A=\hat Q\hat R$，且 $\Vert\Delta A\Vert/\Vert A\Vert=O(u)$。算法没有把 rounding 变成远离原问题的另一个问题。

**Q2.** 为什么正交变换帮得上忙？

**A.** 每步 $\Vert H_k\Vert_2=1$，前面的 perturbation 不会在后续反射里被指数放大。

**Q3.** QR stable 是否等于 least squares accurate？

**A.** 不等于。$A$ 接近亏秩时 $R$ 对角会出现极小值。QR 可能准确地报告坏条件，solve 仍然敏感。

**Q4.** 该监控 $\Vert\Delta A\Vert$ 还是 $\Vert\hat x-x\Vert$？

**A.** 先看 backward residual / $\Vert A-\hat Q\hat R\Vert$ 判断分解；再单独看 $\kappa(R)$ 判断后续 solve。


# Lecture 17 — Stability of Back Substitution

### 1. Back substitution

对 upper triangular $R$：

$$
Rx=b.
$$

最后一行先得到

$$
x_n=b_n/r_{nn},
$$

然后递推

$$
x_i=\frac{1}{r_{ii}}
\left(b_i-\sum_{j=i+1}^nr_{ij}x_j\right).
$$

### 2. Numerical risk

两个地方最敏感：

1. $r_{ii}$ 很小：division 放大误差；
2. 括号中的 subtraction cancellation。

对一个 backward-stable triangular solver，可以把计算结果解释为

$$
(R+\Delta R)\hat x=b,
\qquad
|\Delta R|\lesssim O(u)|R|.
$$

但如果 $R$ condition number 大，forward error 依旧可能很大。

### 3. 实验：极端病态的 FP32 triangular solve

#### 3.1 实验目的

本实验只回答一个问题：

> **如果 triangular solve 本身 backward stable，但 triangular factor $R$ 极端病态，FP32 算出的 solution 是否仍然可信？**

实验直接从上三角系统

$$
Rx=b
$$

开始，**不执行 QR 分解**。这样可以把误差来源限制在 FP32 输入舍入与 back substitution，而不混入 factorization 的误差。

它对应完整 QR solver 的最后一步：

$$
A=QR,
\qquad
Ax=b_{\mathrm{original}}
\quad\Longrightarrow\quad
Rx=Q^Tb_{\mathrm{original}}.
$$

因此实验代码中的 `b`，对应完整 QR 流程中已经变换过的右端项 $Q^Tb_{\mathrm{original}}$。

#### 3.2 构造测试矩阵

首先构造 $12\times12$ 的随机上三角矩阵：

```python
R = np.triu(r.normal(size=(n, n)))
```

对角线上方的元素通常为 $O(1)$。随后把对角元设为从 $1$ 到 $10^{-7}$ 的等比数列：

```python
np.fill_diagonal(R, np.geomspace(1, 1e-7, n))
```

这会制造一串越来越小的 pivots。必须注意：

> **最小对角元为 $10^{-7}$，不代表 $\kappa_2(R)=10^7$。**

只有 diagonal matrix 的对角元才直接等于 singular values。这里的 $O(1)$ 非对角元与小 pivots 共同作用，使 $R^{-1}$ 的元素在回代递推中逐层增长。不同 LAPACK 环境下估计出的 $\kappa_2(R)$ 大约在 $10^{24}$–$10^{26}$；具体数字已经不可靠，但“远超任何常用精度可分辨的范围”这个结论不变。

#### 3.3 构造已知答案的问题

为了能够计算 forward error，先生成已知参考解 $x_{\mathrm{true}}$，再构造右端项：

$$
b=Rx_{\mathrm{true}}.
$$

于是 float64 中的参考问题具有已知答案 $x_{\mathrm{true}}$。接着把 $R$ 和 $b$ 舍入到 FP32：

$$
R_{32}=\operatorname{fl}_{32}(R),
\qquad
b_{32}=\operatorname{fl}_{32}(b),
$$

并通过 back substitution 求解

$$
R_{32}\hat x_{32}=b_{32}.
$$

这里把 $\hat x_{32}$ 再 cast 回 float64 只为了计算误差；已经在 FP32 中丢失的信息不会因此恢复。

#### 3.4 先定义 backward error

对于一般问题

$$
y=f(d),
$$

forward error 问的是：**计算结果 $\hat y$ 离正确答案 $y$ 多远？**

backward error 问的是另一个问题：

> **最少需要把输入数据 $d$ 改动多少，才能把计算结果 $\hat y$ 看成修改后问题的精确答案？**

也就是寻找最小的输入扰动 $\Delta d$，使

$$
\hat y=f(d+\Delta d).
$$

因此 backward error 衡量的是“算法实际解了一个离原问题多远的问题”，而不是“答案本身错了多少”。

对本实验的线性系统，输入数据是矩阵与右端项这一对 $(R,b)$。给定计算结果 $\hat x$，定义 residual

$$
r=b-R\hat x.
$$

允许矩阵和右端项同时发生相对大小不超过 $\epsilon$ 的扰动：

$$
(R+\Delta R)\hat x=b+\Delta b,
$$

其中

$$
\Vert\Delta R\Vert_2\le\epsilon\Vert R\Vert_2,
\qquad
\Vert\Delta b\Vert_2\le\epsilon\Vert b\Vert_2.
$$

**normwise relative backward error** 定义为满足这些条件的最小 $\epsilon$：

$$
\eta(\hat x)
=
\min\left\{
\epsilon:
(R+\Delta R)\hat x=b+\Delta b
\right\}.
$$

由修改后的方程可得

$$
r=b-R\hat x=\Delta R\hat x-\Delta b.
$$

因此

$$
\Vert r\Vert_2
\le
\Vert\Delta R\Vert_2\Vert\hat x\Vert_2+
\Vert\Delta b\Vert_2
\le
\epsilon
\left(
\Vert R\Vert_2\Vert\hat x\Vert_2+
\Vert b\Vert_2
\right).
$$

取能够解释 residual 的最小扰动，在 2-norm 下得到

$$
\boxed{
\eta(\hat x)=
\frac{\Vert R\hat x-b\Vert_2}
{\Vert R\Vert_2\Vert\hat x\Vert_2+\Vert b\Vert_2}
}.
$$

这正是下面代码计算的 `backward_error`。分母不是任意选择的：它表示当 $R$ 和 $b$ 各自允许相对改动 $\epsilon$ 时，最多能解释多大的 residual。

如果要求 $b$ 完全不变，只允许修改 $R$，相应定义则是

$$
\eta_R(\hat x)
=
\min_{(R+\Delta R)\hat x=b}
\frac{\Vert\Delta R\Vert_2}{\Vert R\Vert_2}
=
\frac{\Vert R\hat x-b\Vert_2}
{\Vert R\Vert_2\Vert\hat x\Vert_2}.
$$

back substitution 的经典 backward-stability 结论更强，它把计算结果解释为

$$
(R+\Delta R)\hat x=b,
\qquad
|\Delta R|\le\gamma_n|R|,
\qquad
\gamma_n\approx n u,
$$

其中 $u$ 是 unit roundoff，$n$ 是矩阵维数。因此所谓 **backward stable**，是指对一类输入都能证明 backward error 至多为 $O(u)$（允许依赖维数的常数），而不是仅凭一次实验中 backward error 很小就下结论。

#### 3.5 实验测量的四个量

1. **矩阵条件数**
   $$
   \kappa_2(R)=\Vert R\Vert_2\Vert R^{-1}\Vert_2.
   $$
   它衡量问题本身对输入扰动的 sensitivity。

2. **relative forward error**
   $$
   \frac{\Vert\hat x_{32}-x_{\mathrm{true}}\Vert_2}
   {\Vert x_{\mathrm{true}}\Vert_2}.
   $$
   它直接回答“算出的解离正确答案多远”。

3. **relative residual**
   $$
   \frac{\Vert R\hat x_{32}-b\Vert_2}{\Vert b\Vert_2}.
   $$
   它只用 $\Vert b\Vert_2$ 归一化，并不直接等于 backward error。

4. **normwise relative backward error**
   $$
   \eta(\hat x_{32})=
   \frac{\Vert R\hat x_{32}-b\Vert_2}
   {\Vert R\Vert_2\Vert\hat x_{32}\Vert_2+\Vert b\Vert_2}.
   $$
   它回答“输入问题最少需要相对改动多少，$\hat x_{32}$ 才会成为精确解”。

#### 3.6 结果应该按什么顺序解释

**第一步：先看 condition number。** 本例的 $\kappa_2(R)$ 约为 $10^{24}$–$10^{26}$，说明问题本身已经极端敏感。这不是普通 FP32 系统，而是 stress test。

**第二步：与工作精度比较。** FP32 的 unit roundoff 为

$$
u\approx6\times10^{-8},
$$

所以

$$
\kappa_2(R)u\gg1.
$$

因此 FP32 无法保证任何正确有效数字；巨大的 forward error 是 problem conditioning 的预期结果。即使换成 FP64，$\kappa_2(R)u$ 仍远大于 $1$。

**第三步：看 forward error。** 输出中的 relative forward error 极大，确认 $\hat x_{32}$ 不能作为参数解使用。

**第四步：最后比较 residual 与 backward error。** relative residual 可以很大，但 normwise backward error 仍约为 $10^{-9}$。这说明本次计算结果是某个 nearby problem 的精确解，与 back substitution 的 backward-stability 定理一致；真正导致 solution 崩溃的是 $R$ 的极端 conditioning。一次实验本身不能证明算法对所有输入都 backward stable。

#### 3.7 实验结论

> **back substitution 可以 backward stable，同时 forward solution 完全错误。两者并不矛盾：前者描述算法引入的扰动，后者还要乘上问题本身的 sensitivity。**

这个实验不是要证明“FP32 不适合 triangular solve”，而是要证明：

$$
\text{small backward error}
\not\Rightarrow
\text{small forward error}
$$

除非问题本身 well-conditioned。

### 4. ML systems mapping

QR、Cholesky 和 LU 最终都要执行 triangular solve：

- QR：解 $Rx=Q^Tb$；
- Cholesky：依次解 $Ly=b$ 与 $L^Tx=y$；
- LU：依次解 $Ly=Pb$ 与 $Ux=y$。

因此 “factorization 成功”只说明得到了 factor，并不保证最终参数解可靠。还必须检查 triangular factor 的 condition number、forward-sensitive directions，以及实际工作精度是否满足 $\kappa u\ll1$。

In [11]:
r = np.random.default_rng(10)
n = 12
R = np.triu(r.normal(size=(n, n)))
np.fill_diagonal(R, np.geomspace(1, 1e-7, n))

R_pretty = np.array2string(
    R,
    formatter={"float_kind": lambda value: f"{value:10.3e}"},
    max_line_width=200,
)
print("R =")
print(R_pretty)
print("diag(R) =", np.array2string(np.diag(R), precision=3, suppress_small=False))

x_true = r.normal(size=n)
b = R @ x_true
R32 = R.astype(np.float32)
b32 = b.astype(np.float32)

x32 = np.linalg.solve(R32, b32).astype(float)

residual_norm = np.linalg.norm(R @ x32 - b)
relative_residual = residual_norm / np.linalg.norm(b)
backward_error = residual_norm / (
    np.linalg.norm(R, 2) * np.linalg.norm(x32) + np.linalg.norm(b)
)

print(f'cond_2(R)                = {np.linalg.cond(R):.2e}')
print(f'FP32 unit roundoff       = {np.finfo(np.float32).eps / 2:.2e}')
print(f'FP32 relative forward err= {relerr(x32, x_true):.2e}')
print(f'relative residual / ||b||= {relative_residual:.2e}')
print(f'normwise backward error  = {backward_error:.2e}')

R =
[[ 1.000e+00 -7.250e-01 -7.818e-01  2.670e-01 -2.486e-01  1.265e-01  8.430e-01  8.579e-01  4.752e-01 -4.508e-01 -7.549e-01 -8.148e-01]
 [ 0.000e+00  2.310e-01 -9.723e-01 -1.134e+00  3.057e-01 -1.852e+00 -1.771e-01  4.258e-01 -9.854e-01 -1.113e+00 -7.606e-01  6.480e-01]
 [ 0.000e+00  0.000e+00  5.337e-02  1.014e+00  9.837e-01  6.300e-01 -2.381e-01 -1.845e+00  1.696e-01 -1.760e-01  7.680e-02  1.542e+00]
 [ 0.000e+00  0.000e+00  0.000e+00  1.233e-02 -6.644e-01 -7.374e-01  7.670e-01  5.046e-01 -4.895e-01  1.153e+00  1.843e-01 -1.340e+00]
 [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00  2.848e-03 -3.923e-01  5.899e-01 -2.192e+00 -1.277e+00 -4.245e-01  2.492e-01 -6.651e-01]
 [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00  6.579e-04 -2.974e-01 -3.820e-01 -9.090e-01  6.117e-01  8.046e-01 -5.642e-01]
 [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00  1.520e-04 -5.052e-01 -1.892e-01  5.194e-01 -1.248e-01 -6.340e-01]
 [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.00

**ML numerics 自测**

**Q1.** 回代 $Rx=b$ 哪两步最危险？

**A.** $x_n=b_n/r_{nn}$：小 $r_{ii}$ 的除法放大误差；括号里 $b_i-\sum r_{ij}x_j$ 的 subtraction cancellation。

**Q2.** backward-stable triangular solve 实际在解什么？

**A.** $(R+\Delta R)\hat x=b$，且 $|\Delta R|\lesssim O(u)|R|$。$R$ 的 $\kappa$ 大时 forward error 仍可很大。

**Q3.** factorization 成功是否等于整个 solve 安全？

**A.** 不等于。Cholesky/QR/LU 最后都落到 triangular solve。分解成功只走完前半段。

**Q4.** ML 小系统 solve 该盯哪个量？

**A.** $|r_{ii}|$、$\kappa(R)$、residual，以及相对 FP64 的 forward error。


# Lecture 18 — Conditioning of Least Squares Problems

### 1. Least squares conditioning 比 square solve 更丰富

$$
x_*=(A^TA)^{-1}A^Tb=A^+b
$$

（full column rank）。如果 $\sigma_{\min}(A)$ 很小，$A^+$ norm 很大：

$$
\Vert A^+\Vert_2=1/\sigma_{\min}(A).
$$

### 2. Residual angle

令 fitted vector $p=Ax_*$，residual $r=b-p$。当 $b$ 几乎 orthogonal to column space 时，$p$ 很小而 residual 很大，relative parameter sensitivity 也会恶化。

因此 least-squares conditioning 不仅取决于 $\kappa(A)$，还取决于 $b$ 与 $\mathcal R(A)$ 的几何关系。

### 3. ML interpretation：什么是 identifiability？

对 linear probe

$$
y\approx Xw,
$$

**parameter identifiability（参数可辨识性）**问的是：

> 给定观测到的 $X$ 与 prediction，参数 $w$ 能否被唯一而稳定地确定？

#### 3.1 Exact non-identifiability

如果存在非零方向 $v$ 满足

$$
Xv=0,
$$

那么对任意标量 $\alpha$，

$$
X(w+\alpha v)=Xw.
$$

因此 $w$ 与 $w+\alpha v$ 给出完全相同的 prediction 和 loss。数据无法判断 $v$ 方向上的 coefficient 应该是多少；这叫 **exact non-identifiability**。

#### 3.2 Practical non-identifiability

如果没有严格的 null direction，但某个单位向量 $v$ 对应很小的 singular value：

$$
\Vert Xv\Vert_2=\sigma_{\min}(X)\ll1,
$$

那么参数沿 $v$ 改变很多，prediction 仍只改变一点：

$$
\Vert X(w+\alpha v)-Xw\Vert_2
=|\alpha|\,\sigma_{\min}(X).
$$

此时 least-squares 解在数学上可能仍然唯一，但对数据噪声、rounding error 或 checkpoint 变化非常敏感。这叫 **practical non-identifiability** 或 weak identifiability。

#### 3.3 它与 large residual 是两个不同问题

需要区分：

1. target 的很大一部分不在 $\mathcal R(X)$ 中，导致无法拟合的 residual 很大；
2. $X$ 存在 null/near-null directions，导致参数 $w$ 无法被唯一或稳定地确定。

两种现象可以同时出现，但 **large residual 并不会自动推出 parameter non-identifiability**。后者主要由 $X$ 的 rank 和 singular values 决定。

因此可能出现：两个 checkpoint 的 prediction 与 loss 几乎相同，但 probe weight 相差很大。原因是两组 weight 的差主要落在 $X$ 看不见或几乎看不见的方向上。

**Representation identifiability** 是相似但更广的概念：模型内部 representation 是否能从其行为中被唯一确定。例如同时对 embedding 做可逆 basis change、对下一层 weight 做逆变换，模型输出可以完全不变。因此跨 checkpoint 直接比较 representation coordinates 或 raw weights，通常还需要先处理 rotation、permutation、scaling 等等价变换。

所以应分别报告：

- **prediction stability**：模型输出是否稳定；
- **parameter identifiability**：参数是否被数据唯一且稳定地约束；
- 若比较 hidden features，再讨论 **representation identifiability**。

**ML numerics 自测**

**Q1.** least squares 的 $\Vert A^+\Vert$ 由什么决定？

**A.** full column rank 时 $x_*=A^+b$，$\Vert A^+\Vert_2=1/\sigma_{\min}(A)$。小 $\sigma_{\min}$ 让参数对扰动极敏感。

**Q2.** conditioning 是否只取决于 $\kappa(A)$？

**A.** 不只。还取决于 $b$ 与 $\mathcal{R}(A)$ 的夹角：residual 大、$p=Ax_*$ 小时，相对参数灵敏度更差。

**Q3.** linear probe 为何能 loss 相近、参数却完全不同？

**A.** 如果 $X$ 存在 null/near-null direction $v$，则 $Xv=0$ 或 $\Vert Xv\Vert\ll1$。参数可以沿 $v$ 改变很多，而 prediction 和 loss 几乎不变。这是 parameter non-identifiability，与 target 是否位于 embedding span 中是不同问题。

**Q4.** 比较两个 checkpoints 的 linear probe 时，如果它们的 prediction 和 loss 很接近，能否据此断定它们学到了相同的 probe parameters 或 hidden representations？还需要分别检查什么？

**A.** 不能。prediction 接近只说明两者在观测数据上的输出接近。还应分别检查：

1. **prediction stability**：输入发生扰动或更换数据后，输出是否仍然接近；
2. **parameter identifiability**：$X$ 是否存在 null/near-null directions，使不同的 probe weights 产生近似相同的 prediction；
3. **representation identifiability**：比较 hidden features 时，是否已经处理 rotation、permutation、scaling 等不改变模型行为的等价变换。


# Lecture 19 — Stability of Least Squares Algorithms

### 1. Normal equations 为什么危险

$$
A^TAx=A^Tb.
$$

SVD：

$$
A=U\Sigma V^T
\Rightarrow
A^TA=V\Sigma^2V^T.
$$

因此

$$
\kappa_2(A^TA)
=\frac{\sigma_1^2}{\sigma_n^2}
=\kappa_2(A)^2.
$$

这不是小常数差异，而是**把 condition number 平方**。

### 2. 三种常见路线

**Normal equations**

$$
A^TAx=A^Tb
$$

便宜、结构简单，但失去精度。

**QR**

$$
A=QR,\qquad Rx=Q^Tb
$$

通常是 dense least squares 的稳定默认。

**SVD**

$$
x=V\Sigma^+U^Tb
$$

最能处理 rank deficiency，也最贵。

### 3. 工程判断

如果你是在做 calibration / fitting，并且 feature covariance spectrum 很差，不要只因为 $A^TA$ 容易写就默认 normal equations。

In [6]:
m,n=120,12
z=rng.normal(size=(m,1))
A=np.hstack([z+10**(-j/2)*rng.normal(size=(m,1)) for j in range(n)])
xt=rng.normal(size=n)
b=A@xt+1e-8*rng.normal(size=m)
xq,*_=np.linalg.lstsq(A,b,rcond=None)
AtA=(A.astype(np.float32).T@A.astype(np.float32)).astype(np.float32)
Atb=(A.astype(np.float32).T@b.astype(np.float32)).astype(np.float32)
try:
    xn=np.linalg.solve(AtA,Atb).astype(float)
    ne=relerr(xn,xt)
except np.linalg.LinAlgError: ne=np.inf
print(f'cond(A) {np.linalg.cond(A)} cond(A^TA) {np.linalg.cond(A.T @ A)}')
print(f'lstsq parameter error {relerr(xq, xt)}')
print(f'FP32 normal equations error {ne}')

cond(A) 501738.5469203536 cond(A^TA) 251741460596.51672
lstsq parameter error 2.253782575384899e-05
FP32 normal equations error 1.7495261646509712


**ML numerics 自测**

**Q1.** normal equations 最关键的数值罪行是什么？

**A.** $\kappa_2(A^TA)=\kappa_2(A)^2$。不是小常数，是把条件数平方。

**Q2.** 三条路线怎么选？

**A.** normal equations：便宜但不稳。QR：$A=QR$、$Rx=Q^Tb$，dense LS 的默认稳定选择。SVD：$x=V\Sigma^+U^Tb$，最能处理亏秩，也最贵。

**Q3.** feature covariance spectrum 很差时，为什么不要图省事写 $A^TA$？

**A.** calibration / fitting 时 $A^TA$ 好写，但精度先死在平方后的 $\kappa$。

**Q4.** 该盯 $\kappa(A)$ 还是 $\kappa(A^TA)$？

**A.** 两者都看。若你走了 normal equations，真正决定参数误差的是 $\kappa(A^TA)$。
